#use llamafactory environment

In [1]:
from datasets import load_dataset
import pandas as pd
import json


/n/home07/than157/.conda/envs/llamafactory/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#load training set of hellaswag
dataset = load_dataset("allenai/winogrande", name="winogrande_xl", split="train")

#print dataset info
print("Dataset info:")
print(dataset)

print("# samples:", len(dataset)) #should be 39k
n_samples = len(dataset)

#convert to dataframe
df = dataset.to_pandas()
df.head()


Dataset info:
Dataset({
    features: ['sentence', 'option1', 'option2', 'answer'],
    num_rows: 40398
})
# samples: 40398


,sentence,option1,option2,answer
0,Ian volunteered to eat Dennis's menudo after a...,Ian,Dennis,2
1,Ian volunteered to eat Dennis's menudo after a...,Ian,Dennis,1
2,"He never comes to my home, but I always go to ...",home,house,1
3,"He never comes to my home, but I always go to ...",home,house,2
4,"Kyle doesn't wear leg warmers to bed, while Lo...",Kyle,Logan,2


In [3]:
# add 'correct_word' using Winogrande's answer key:
# answer == '1' -> option1, answer == '2' -> option2
df["correct_word"] = df.apply(
    lambda row: row["option1"] if str(row["answer"]) == "1" else row["option2"],
    axis=1,
)
df.head(50)

,sentence,option1,option2,answer,correct_word
0,Ian volunteered to eat Dennis's menudo after a...,Ian,Dennis,2,Dennis
1,Ian volunteered to eat Dennis's menudo after a...,Ian,Dennis,1,Ian
2,"He never comes to my home, but I always go to ...",home,house,1,home
3,"He never comes to my home, but I always go to ...",home,house,2,house
4,"Kyle doesn't wear leg warmers to bed, while Lo...",Kyle,Logan,2,Logan
5,"Kyle doesn't wear leg warmers to bed, while Lo...",Kyle,Logan,1,Kyle
6,The GPS and map helped me navigate home. I go...,GPS,map,2,map
7,The GPS and map helped me navigate home. I go...,GPS,map,1,GPS
8,Emily looked up and saw Patricia racing by ove...,Emily,Patricia,2,Patricia
9,Emily looked up and saw Patricia racing by ove...,Emily,Patricia,1,Emily


In [4]:
#create 'sentence_prefix' and 'sentence_suffix' columns
df["sentence_prefix"] = df["sentence"].str.split("_").str[0]
df["sentence_suffix"] = df["sentence"].str.split("_").str[1]

#remove whitespace from 'sentence_prefix' and 'sentence_suffix'
df["sentence_prefix"] = df["sentence_prefix"].str.rstrip() #remove trailing space
df["sentence_suffix"] = df["sentence_suffix"].str.lstrip() #remove leading space

df.head()


,sentence,option1,option2,answer,correct_word,sentence_prefix,sentence_suffix
0,Ian volunteered to eat Dennis's menudo after a...,Ian,Dennis,2,Dennis,Ian volunteered to eat Dennis's menudo after a...,despised eating intestine.
1,Ian volunteered to eat Dennis's menudo after a...,Ian,Dennis,1,Ian,Ian volunteered to eat Dennis's menudo after a...,enjoyed eating intestine.
2,"He never comes to my home, but I always go to ...",home,house,1,home,"He never comes to my home, but I always go to ...",is smaller.
3,"He never comes to my home, but I always go to ...",home,house,2,house,"He never comes to my home, but I always go to ...",is bigger.
4,"Kyle doesn't wear leg warmers to bed, while Lo...",Kyle,Logan,2,Logan,"Kyle doesn't wear leg warmers to bed, while Lo...",is more likely to live in a colder climate.


In [5]:
#print first 50 rows, showing the full contents of each column -- check that the data is correct
print(df.head(50).to_string())

                                                                                                                                                           sentence      option1    option2 answer correct_word                                                                                                                       sentence_prefix                              sentence_suffix
0                                                           Ian volunteered to eat Dennis's menudo after already having a bowl because _ despised eating intestine.          Ian     Dennis      2       Dennis                                                            Ian volunteered to eat Dennis's menudo after already having a bowl because                   despised eating intestine.
1                                                            Ian volunteered to eat Dennis's menudo after already having a bowl because _ enjoyed eating intestine.          Ian     Dennis      1          Ian                   

In [6]:
#shuffle rows since rows are in pairs
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.head()


,sentence,option1,option2,answer,correct_word,sentence_prefix,sentence_suffix
0,Monica confided in Cynthia that she had prepar...,Monica,Cynthia,2,Cynthia,Monica confided in Cynthia that she had prepar...,was impressed.
1,Rebecca had less grass in their yard than Elen...,Rebecca,Elena,1,Rebecca,Rebecca had less grass in their yard than Elen...,had more rabbits in their yard.
2,Erin has a lot of dry skin on their arms and l...,Erin,Angela,2,Angela,Erin has a lot of dry skin on their arms and l...,has been using lotion to keep their skin soft.
3,Joel really enjoyed shopping and Hunter did no...,Joel,Hunter,1,Joel,Joel really enjoyed shopping and Hunter did no...,intended to be a personal shopper.
4,The spouse of Logan cheated on him with Brett ...,Logan,Brett,1,Logan,The spouse of Logan cheated on him with Brett ...,always ignored her feelings.


In [7]:
### create json file

#format data for sft
data = []

for idx, row in df.iterrows():
    item = {
        "instruction": f'{row["sentence_prefix"]} {row["correct_word"]}',
        "input": "",
        "output": row["sentence_suffix"]
    }
    data.append(item)
    #track progress
    if idx % 10000 == 0:
        print(f"Processed {idx} rows")
    if idx == (n_samples - 1):
        print(f"Processed {idx} rows (last row)")

#save to JSON file
with open("data/winogrande.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print("Complete!")

Processed 0 rows
Processed 10000 rows
Processed 20000 rows
Processed 30000 rows
Processed 40000 rows
Processed 40397 rows (last row)
Complete!
